# The ESEM sandbox, one year at a time

This model asks who would build a power station, rather than what a system ought to contain. Each firm faces an income it cannot predict, values that income below its average because it is uncertain, and commits only when what it expects to earn covers what the plant costs to own. No one in it is obliged to build anything.

A least-cost model answers the other question, and the distance between the two answers is what this notebook is about. It follows one year of the model in the order the model runs it, the same order as [HOW_IT_WORKS.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/HOW_IT_WORKS.md), and every section ends by changing one setting and comparing. The last section is a dashboard over a comparison you configure.

Everything here is illustrative. The fleet is stylised, the weather is synthetic, and the system is one region with no transmission in it.

The runs below are shorter than the model's own defaults so that a cell finishes while you are looking at it: 18 possible futures rather than 45, and eight years rather than 20. Totals here are not comparable with the 20-year figures in the repository, and the sign of a result can change on another weather draw.

In [ ]:
!pip install -q git+https://github.com/MarkusMannheim/esem_sandbox.git  # skip if you have it

from esem_sandbox.config import load_settings
from esem_sandbox.core.forward import cell_plan
from esem_sandbox.core.simulate import run, MERCHANT, ESEM
from esem_sandbox import explore, plots
import numpy as np

settings = load_settings()
TICKS, SEED = 8, 20260904
QUICK = explore.quick_cells(settings)     # two of the five weather years
print(f'{len(cell_plan(settings))} futures in the full lattice, {len(QUICK)} here')


def pair(overrides=None, **options):
    """Both legs on one weather draw, on the quick lattice. ``overrides`` is a
    dictionary of settings sections, as in PARAMETERS.md."""
    s = load_settings(overrides)
    opts = dict(ticks=TICKS, seed=SEED, quick=True)
    opts.update(options)
    return s, explore.run_pair(s, **opts)


def table(named):
    """Side by side, one column per pair: what a comparison turns on. The two
    moves are merchant less scheme, so a positive number means the scheme
    lowered that line."""
    rows = (('unserved, merchant, GWh', 'merchant_unserved_gwh', 1, 2),
            ('unserved, with the scheme, GWh', 'esem_unserved_gwh', 1, 2),
            ('bill move, $bn', 'bill_move', 1e-9, 2),
            ('resource cost move, $bn', 'resource_cost_move', 1e-9, 2),
            ('new plant, merchant, MW', 'merchant_built_mw', 1, 0),
            ('new plant, scheme, unsubsidised, MW', 'esem_built_mw', 1, 0),
            ('awarded by the scheme, MW', 'awarded_mw', 1, 0),
            ('levy over the run, $bn', 'levy', 1e-9, 2))
    summaries = {k: explore.summarise(s, legs) for k, (s, legs) in named.items()}
    width = max(18, max(len(k) for k in named) + 2)
    print(f"{'':<38}" + ''.join(f'{k:>{width}}' for k in named))
    for label, key, scale, dp in rows:
        print(f'{label:<38}' + ''.join(f'{summaries[k][key] * scale:>{width},.{dp}f}'
                                      for k in named))

## 1. The hour: how a price is made

Plant is stacked cheapest first and the price is set by the last unit needed. In most hours that is tens of dollars. In a handful it is thousands, and those few hours carry most of the year's price and almost all of a peaker's income. The cell below prints the stack the model prices an hour from, then dispatches the stressed weather year, the one with a wind lull over a heatwave, and counts those hours.

In [ ]:
from esem_sandbox.core.dispatch import dispatch_year
from esem_sandbox.core.weather import generate_bundle

bundle = generate_bundle(settings.weather['seed'], settings.weather['shape_years'])
STRESS = 4                                  # the lull-on-heat year
PEAK = 12_500.0                             # MW, the first year's system peak


def dispatch(s, shape_year=STRESS):
    shape = bundle['demand_shape'][shape_year]
    return dispatch_year(s, 2026, shape * (PEAK / shape.max()),
                         bundle['wind_cf'][shape_year], bundle['solar_cf'][shape_year])


print(f"{'offer, $/MWh':>14}  {'capacity, MW':>12}  who")
stack = sorted([(u.srmc_per_mwh, u.available_mw, u.unit) for u in settings.fleet
                if u.technology not in ('rooftop', 'hydro') and not u.duration_h]
               + [(t.price_per_mwh, t.capacity_mw, f'demand response {t.tier}')
                  for t in settings.dsr])
for offer, mw, who in stack:
    print(f'{offer:>14,.0f}  {mw:>12,.0f}  {who}')
print(f"{settings.market['market_price_cap_per_mwh']:>14,.0f}  {'':>12}  the price cap")

year = dispatch(settings)
hours_above = int((year.price >= 300).sum())
print(f'\n{hours_above} hours of 8,760 priced at or above $300/MWh')
print(f'they carry {year.price[year.price >= 300].sum() / year.price.sum():.0%} of the year total')
print(f'unserved energy {year.unserved_mwh.sum() / 1000:.3f} GWh')

In [ ]:
plots.price_duration({'the lull-on-heat year': year}, 'duration.png')
from IPython.display import Image
Image('duration.png')

Change one thing. The import link in the fleet table is the lever that sets how tight the system is. Shrink it and dispatch the same year again.

In [ ]:
from dataclasses import replace

for import_mw in (1000.0, 800.0, 600.0):
    fleet = tuple(replace(u, capacity_mw=import_mw) if u.technology == 'import' else u
                  for u in settings.fleet)
    res = dispatch(replace(settings, fleet=fleet))
    print(f'import link {import_mw:>6,.0f} MW   unserved {res.unserved_mwh.sum() / 1000:>7.3f} GWh   '
          f'hours at or above $300: {int((res.price >= 300).sum()):>3}')

## 2. The year: settlement, and how much gets traded

Nobody in the model is given a contract position. Two retailers hedge to a mandate, buying a strip every year so that overlapping strips carry their target; four producers write what they have, pro rata to capacity. What each producer has sold forward is its cover, and cover is what lowers its own bar in section 4. The run below is the market on its own; the table gives, for each year, how many contracts are in force and how much of each producer's output is sold forward.

In [ ]:
base = run(settings, ticks=TICKS, seed=SEED, cells=QUICK, leg=MERCHANT)
producers = list(base.ticks[0].swap_cover)
print(f"{'year':>6}{'contracts':>11}" + ''.join(f'{p:>20}' for p in producers))
for t in base.ticks:
    print(f'{t.year:>6}{t.live_contracts:>11}'
          + ''.join(f'{100 * t.swap_cover[p]:>19.0f}%' for p in producers))
print('\ncover is each producer\'s output sold forward under swaps and contracts for difference')

Change one thing. By default both sides accept a published reference price and trade the whole volume. Under the crossing option each side steps away from that price and they trade only what both wanted, which is a different amount of cover and so a different bar.

In [ ]:
crossed = run(settings, ticks=TICKS, seed=SEED, cells=QUICK, leg=MERCHANT,
              clearing='crossing')
last, last_x = base.ticks[-1], crossed.ticks[-1]
print(f"{'':<22}{'anchor':>10}{'crossing':>10}")
print(f"{'contracts in force':<22}{last.live_contracts:>10}{last_x.live_contracts:>10}")
for p in producers:
    print(f'{p + " cover":<22}{100 * last.swap_cover[p]:>9.0f}%{100 * last_x.swap_cover[p]:>9.0f}%')
print(f"{'new plant, MW':<22}{base.total_built_mw:>10,.0f}{crossed.total_built_mw:>10,.0f}")

## 3. The forward view: possible futures at three distances

No one in the model forecasts a price. Each year it dispatches every one of its possible futures, at fixed odds, four, eight and twelve years ahead, and hands each firm a spread of what a megawatt of each technology would earn. The cell below builds the view an investor reads at the start of the run and prints the spread for each technology at the nearest distance, against what the plant costs to own.

In [ ]:
from esem_sandbox.core.esem import long_run_cost_per_mw_year
from esem_sandbox.core.forward import EntryState, forward_view
from esem_sandbox.core.simulate import with_tail

view = with_tail(settings, forward_view(settings, base.fleet, bundle, year=2026,
                                        peak_mw=PEAK, entry=EntryState(),
                                        cells=QUICK), base.roster)
near = view.anchors[0]
print(f'{len(near.outcomes)} futures at {near.offset} years ahead, $000 per MW-year\n')
print(f"{'technology':<12}{'lowest':>10}{'expected':>10}{'highest':>10}{'fixed cost':>12}")
for tech in settings.tech_costs:
    rents = near.rents(tech.technology) / 1000
    fixed = long_run_cost_per_mw_year(tech, tech.wacc, 0.0) / 1000
    print(f'{tech.technology:<12}{rents.min():>10,.0f}{near.expected(rents):>10,.0f}'
          f'{rents.max():>10,.0f}{fixed:>12,.0f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.2), facecolor=plots.SURFACE)
plots._style(ax)
tech = settings.tech('ocgt')
for i, anchor in enumerate(view.anchors):
    rents = anchor.rents('ocgt') / 1000
    ax.scatter([anchor.offset] * len(rents), rents, s=22, color=plots.SERIES[2], alpha=0.55,
               label='one possible future' if i == 0 else None)
    ax.scatter([anchor.offset], [anchor.expected(rents)], s=70, color=plots.SERIES[0],
               zorder=3, label='the expectation, at the odds' if i == 0 else None)
fixed = long_run_cost_per_mw_year(tech, tech.wacc, 0.0) / 1000
ax.axhline(fixed, color=plots.INK_MUTED, ls='--', lw=1.2)
ax.text(view.anchors[0].offset - 0.4, fixed, 'what a peaker costs to own', va='bottom',
        fontsize=9, color=plots.INK_2)
ax.set_xticks([a.offset for a in view.anchors])
ax.set_xlabel('years ahead')
ax.set_ylabel('rent a peaker would earn, $000 per MW-year')
plots.titled(ax, 'What a peaker would earn in each possible future')
plots.finish(fig, legend_from=ax, ncol=2)
fig.savefig('futures.png', dpi=130, facecolor=plots.SURFACE)
plt.close(fig)
Image('futures.png')

Change one thing. The odds on the three demand growth paths never move; they come from the growth table. The peak bands scale the peak in a severe, a normal and a mild year. Widen the severe band and the spread widens with it.

In [ ]:
for severe in (1.08, 1.15):
    s = load_settings({'weather': {'peak_band_multipliers': [severe, 1.0, 0.95]}})
    v = with_tail(s, forward_view(s, base.fleet, bundle, year=2026, peak_mw=PEAK,
                                  entry=EntryState(), cells=explore.quick_cells(s)),
                  base.roster)
    rents = v.anchors[0].rents('ocgt') / 1000
    print(f'severe peak at {severe:.2f} x normal: a peaker earns {rents.min():>6,.0f} to '
          f'{rents.max():>6,.0f}, expected {v.anchors[0].expected(rents):>6,.0f} $000 per MW-year')

## 4. The decision: why a firm hesitates

A plant is built when what it expects to earn, per megawatt per year, covers what it costs to own. The firm does not compare the average of its possible incomes with its cost; it compares the certain sum it would swap that spread for, which is lower by an amount that grows with the spread and with the firm's own dislike of risk. Each of the market's producers prices the same peaker below; the gap between what it costs and what each demands is caution.

In [ ]:
from esem_sandbox.core.agents import PRODUCER
from esem_sandbox.core.clearing import cara_certainty_equivalent, cara_coefficient
from esem_sandbox.core.investment import residual_exposure

tech = settings.tech('ocgt')
fixed = long_run_cost_per_mw_year(tech, tech.wacc, 0.0)
# Caution is priced over the three growth paths, each at the mean of its weather
# and peak cells: a run keeps its growth path for life and redraws the other two.
rents, weights = view.risk_distribution(tech, settings)
print(f'a peaker costs {fixed:>12,.0f} $/MW-year to own\n')
for agent in [a for a in base.roster if a.kind == PRODUCER]:
    exposure = residual_exposure(settings, tech.life_years)
    a = cara_coefficient(agent.risk_aversion, exposure, settings)
    ce = cara_certainty_equivalent(rents, weights, a)
    print(f'{agent.name:>20} demands {fixed + max(0.0, rents @ weights - ce):>12,.0f} $/MW-year')

Change one thing. `investment.risk_premium` is the weight on that gap. At zero every firm is risk-neutral and the bar is the cost.

In [ ]:
for premium in (0.0, 0.125, 0.25, 0.5):
    s = load_settings({'investment': {'risk_premium': premium}})
    a = cara_coefficient(0.45, residual_exposure(s, tech.life_years), s)
    ce = cara_certainty_equivalent(rents, weights, a)
    print(f'risk premium {premium:>5.3f}   the regional merchant demands '
          f'{fixed + max(0.0, rents @ weights - ce):>12,.0f} $/MW-year')

## 5. What lowers the bar: contracts, and the scheme's award

Selling output forward removes the uncertainty rather than paying for it, so the bar falls without anyone handing the plant money. The retailers' swap book reaches a few years into a plant's life; the scheme's award reaches twelve, from the plant's fourth year. The first cell prices the same peaker with more and more of its output under an award. The second runs the scheme beside the market on one weather draw and prints, for each year, how much firm capacity the scheme's lane sought, what it awarded, the price the plant bid on top of what it expects to earn, and the levy consumers paid.

In [ ]:
tenor = settings.esem['contract_tenor_years']
start = settings.esem['contract_start_year_of_plant']
for cover in (0.0, 0.5, 1.0):
    exposure = residual_exposure(settings, tech.life_years, award_years=tenor,
                                 award_cover=cover, award_start_year=start)
    a = cara_coefficient(0.45, exposure, settings)
    ce = cara_certainty_equivalent(rents, weights, a)
    print(f'{cover:>4.0%} of output under a {tenor}-year award from year {start}   '
          f'bar {fixed + max(0.0, rents @ weights - ce):>12,.0f} $/MW-year')

In [ ]:
S0, L0 = pair()                              # the baseline pair, reused below
esem = L0[ESEM]
print(f"{'year':>6}{'lane, MW':>10}{'awarded, MW':>13}{'bid, $/MW-year':>16}{'levy, $/MWh':>13}")
for t in esem.ticks:
    mw = sum(a.capacity_mw for a in t.awards)
    bid = (sum(a.price_per_mw_year * a.capacity_mw for a in t.awards) / mw) if mw else float('nan')
    print(f'{t.year:>6}{t.lane_volume_mw:>10,.0f}{mw:>13,.0f}'
          f'{bid:>16,.0f}{t.levy_per_mwh:>13.2f}')
print('\na bid of zero is a plant that needed nothing and was awarded anyway')

Change one thing. Shorten the award, and start it at commissioning instead of the plant's fourth year.

In [ ]:
table({'12 years from year 4': (S0, L0),
       '6 years from year 4': pair({'esem': {'contract_tenor_years': 6}}),
       '12 years from year 1': pair({'esem': {'contract_start_year_of_plant': 1}})})

## 6. Pacing, the bracket and exit

How much a market builds depends on something the model cannot settle: whether investors can see each other. Under the first rule no one observes anyone, so every firm prices its project against a market that does not contain the others' projects. Under the second everyone observes everyone at once, so the first decision of a year removes the scarcity rent the rest were counting on. Real investors are neither, and the two rules bracket the amount built.

In [ ]:
for rule in ('simultaneous', 'sequential'):
    r = run(settings, ticks=TICKS, seed=SEED, cells=QUICK, leg=MERCHANT, investment=rule)
    print(f'{rule:>14}: built {r.total_built_mw:>7,.0f} MW   '
          f'firm {r.ticks[-1].firm_capacity_mw:>7,.0f} MW   '
          f'unserved {r.total_unserved_gwh:>7.2f} GWh')

The bracket matters because reliability is not proportional to capacity. Take firm plant away in small steps and dispatch the same stressed year each time: each megawatt removed exposes more hours than the last one did.

In [ ]:
VARIABLE = {'wind', 'solar', 'rooftop'}
print(f"{'firm capacity':>16}{'unserved, GWh':>16}{'times the base':>16}")
base_gwh = None
for scale in (1.00, 0.97, 0.94, 0.91, 0.85):
    fleet = tuple(u if u.technology in VARIABLE
                  else replace(u, capacity_mw=u.capacity_mw * scale,
                               must_run_mw=u.must_run_mw * scale)
                  for u in settings.fleet)
    res = dispatch(replace(settings, fleet=fleet))
    firm = sum(u.capacity_mw * u.availability for u in fleet if u.technology not in VARIABLE)
    gwh = float(res.unserved_mwh.sum()) / 1000
    base_gwh = base_gwh or gwh
    print(f'{firm:>13,.0f} MW{gwh:>16.2f}{gwh / base_gwh:>15.1f}x')

Change one thing. The annual build ceiling is a choice standing in for supply chains and crews, and every reliability result sits on it. Raise it and read the unserved lines against the baseline.

In [ ]:
table({'ceiling 2 a year': (S0, L0),
       'ceiling 4 a year': pair({'investment': {'concurrent_builds_per_year': 4}})})

## 7. Boom and bust, and what a comparison means

Both legs of the baseline pair drew one weather sequence from one seed, so the difference between them is the mechanism. The table gives each year's unserved energy as a multiple of the reliability standard, so 1.00x is exactly at the standard. Look at the first years: the legs are identical however much the lane bought, because the plant it paid for has not been built yet. A procurement scheme is an instrument about the future and cannot fix a year that arrives before its plant does.

In [ ]:
m, e = L0[MERCHANT], L0[ESEM]
standard = S0.reliability['standard_use_fraction']
print("unserved energy, as a multiple of the reliability standard")
print(f"{'year':>6}{'merchant':>16}{'with the scheme':>18}{'lane, MW':>10}")
for a, b in zip(m.ticks, e.ticks):
    print(f'{a.year:>6}{a.unserved_fraction / standard:>13.2f}x'
          f'{b.unserved_fraction / standard:>17.2f}x{b.lane_volume_mw:>10.0f}')

summary = explore.summarise(S0, L0)
print(f"\nthe bill moves          {summary['bill_move'] / 1e9:>8,.2f} bn   (positive: consumers pay less)")
print(f"the resource cost moves {summary['resource_cost_move'] / 1e9:>8,.2f} bn   (positive: the economy gives up less)")
print(f"the difference is a transfer of {summary['transfer'] / 1e9:,.2f} bn")

Most of a bill is a payment from consumers to producers. Capacity pushes the pool price down and moves money between them without saving any, so a comparison that showed only the bill would report that movement as a benefit. Read both lines.

The dashboard reads the outcome, eight panels in the order the year runs. The market view reads the market that produced it: the time-of-day prices contracts are written on, how much of each producer's output is sold forward, what the forward view expected against what came, the scheme's auction, who the contracts paid, and what was decided each year.

In [ ]:
plots.dashboard(L0, S0, 'dashboard.png')
Image('dashboard.png')

In [ ]:
plots.market_view(L0, S0, 'market_view.png')
Image('market_view.png')

## 8. Explore: change settings and compare

Everything above ran on one weather draw and the package defaults. The cell below is yours: put any settings from [PARAMETERS.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/PARAMETERS.md) in `OVERRIDES`, any run option in `OPTIONS`, and compare against the baseline. `seed` is the weather draw; `investment='sequential'` is the second rule from section 6; `clearing='crossing'` the negotiated market from section 2; `scheme=True` switches the state renewable scheme on; `retire={'coal_b': 2028}` closes a plant early.

In [ ]:
OVERRIDES = {'investment': {'risk_premium': 0.0}}     # risk-neutral firms
OPTIONS = dict(seed=SEED)

S1, L1 = pair(OVERRIDES, **OPTIONS)
table({'baseline': (S0, L0), 'yours': (S1, L1)})
plots.dashboard(L1, S1, 'dashboard_yours.png')
plots.market_view(L1, S1, 'market_view_yours.png')
Image('dashboard_yours.png')

In [ ]:
Image('market_view_yours.png')

A sweep runs both legs at several values of one setting on the same draw, so the rows differ in that one number and in nothing else. Two values below, to keep the cell short; add more, or sweep another setting.

In [ ]:
rows = explore.sweep('esem.contract_tenor_years', [6, 12], ticks=TICKS, seed=SEED, quick=True)
for r in rows:
    print(f"tenor {r['value']:>3}: unserved merchant {r['merchant_unserved_gwh']:>6.2f} GWh, "
          f"scheme {r['esem_unserved_gwh']:>6.2f} GWh; resource cost move "
          f"{r['resource_cost_move'] / 1e9:>6,.2f} bn; levy {r['levy'] / 1e9:>5,.2f} bn")
plots.sweep(rows, 'esem.contract_tenor_years', 'sweep.png')
Image('sweep.png')

Whatever you change, read the chain of cause and effect rather than the size of any number, and run the same change on a second seed before reading a sign. [HOW_IT_WORKS.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/HOW_IT_WORKS.md) says what each step does and which setting moves it; [GLOSSARY.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/GLOSSARY.md) explains the terms; [KNOWN_LIMITATIONS.md](https://github.com/MarkusMannheim/esem_sandbox/blob/main/KNOWN_LIMITATIONS.md) says which results are easy to misread.